In [0]:
# =============================================================
# FINANCIAL-EVIDENCE RETRIEVER
#
# Requires : 10_runtime_config (run first via %run)
# Globals used: AI_SEARCH_INDEX
# Exposes  : _response_to_df(), retrieve_financial_evidence()
#
# Uses the WorkspaceClient REST path because databricks.vector_search
# and databricks.ai_search are blocked by the Databricks import hook
# after a %pip-install-triggered Python restart. The WorkspaceClient
# is always available and calls the same REST endpoint.
# =============================================================

import json
import pandas as pd
from databricks.sdk import WorkspaceClient as _WSClient

# AI_SEARCH_INDEX is in scope from 10_runtime_config via %run.
_vs_client = _WSClient()

FINANCIAL_SECTIONS = [
    "Selected financial ratios",
    "Objective financial observations",
    "Observed financial points",
]

FINANCIAL_DOCUMENT_TYPES = [
    "financial_summary",
    "credit_analyst_note",
]


def _response_to_df(response: dict) -> pd.DataFrame:
    """Convert the raw Vector Search REST API response to a DataFrame."""
    col_names  = [c["name"] for c in response["manifest"]["columns"]]
    data_array = response["result"]["data_array"]
    if not data_array:
        return pd.DataFrame(columns=col_names)
    extra = len(data_array[0]) - len(col_names)
    if extra == 1:
        col_names = col_names + ["score"]
    elif extra > 1:
        col_names = col_names + [f"extra_{i}" for i in range(extra)]
    return pd.DataFrame(data_array, columns=col_names)


def retrieve_financial_evidence(
    borrower_id: str,
    question: str,
    num_results: int = 6,
) -> pd.DataFrame:
    """
    Borrower-scoped financial evidence retriever.

    Returns chunks from financial_summary and credit_analyst_note
    restricted to quantitative risk sections.
    Does not approve or reject credit.
    """
    if not borrower_id or not borrower_id.startswith("B"):
        raise ValueError(f"borrower_id must start with 'B', got: {borrower_id!r}")
    if not question or not question.strip():
        raise ValueError("question must not be empty.")

    response = _vs_client.api_client.do(
        method="POST",
        path=f"/api/2.0/vector-search/indexes/{AI_SEARCH_INDEX}/query",
        body={
            "columns": [
                "chunk_id", "borrower_id", "company_name",
                "document_id", "document_type", "current_section",
                "page_number", "element_type", "chunk_to_retrieve",
            ],
            "query_text": question,
            "filters_json": json.dumps({
                "borrower_id":     borrower_id,
                "document_type":   FINANCIAL_DOCUMENT_TYPES,
                "current_section": FINANCIAL_SECTIONS,
            }),
            "num_results": num_results,
            "query_type":  "HYBRID",
        },
    )

    df = _response_to_df(response)
    if df.empty:
        return df

    assert (df["borrower_id"] == borrower_id).all(), \
        "Cross-borrower retrieval detected."
    assert df["document_type"].isin(FINANCIAL_DOCUMENT_TYPES).all(), \
        "Unexpected document_type in results."
    assert df["current_section"].isin(FINANCIAL_SECTIONS).all(), \
        "Unexpected section in results."
    assert df["page_number"].notna().all(), \
        "Retrieved chunk is missing page_number."

    return df


print("retrieve_financial_evidence() defined.")
print(f"  Index    : {AI_SEARCH_INDEX}")
print(f"  Sections : {FINANCIAL_SECTIONS}")